# 01 · 微调 ProtBERT-BFD,得到 AMP-BERT

在 Veltri 训练集上微调预训练模型 **ProtBERT-BFD**,微调后的模型就是 **AMP-BERT**。
本 notebook 自包含:每一步单独一个 cell,参数就写在用到它的地方。

## 0 · 环境准备
安装一个仍带经典 BERT 分词器的 `transformers` 版本(Colab 自带的太新,无法正确加载 ProtBERT-BFD),并检查 GPU。

> 运行前请先设置:**代码执行程序 → 更改运行时类型 → GPU**。

In [ ]:
%pip install -q transformers==4.46.3 "tokenizers>=0.20,<0.21" sentencepiece

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用设备:', device)   # 期望输出 cuda

## 1 · 导入依赖

In [ ]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, roc_auc_score)

In [ ]:
# 整个 notebook 共用的两个常量
MODEL_NAME = 'Rostlab/prot_bert_bfd'   # 预训练模型(分词器与模型都用它)
MAX_LEN    = 200                        # 序列截断/补齐到的长度

## 2 · 加载训练数据
Veltri 训练集,直接从仓库读取(无需下载/clone)。

In [ ]:
TRAIN_URL = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/all_veltri.csv'
SEED = 0   # 打乱顺序的随机种子

df = pd.read_csv(TRAIN_URL, index_col=0).sample(frac=1, random_state=SEED)
print('训练样本数:', len(df))
df.head()

## 3 · 把序列编码成模型输入
先定义数据集类,再加载分词器,最后构建训练用的 `train_dataset`。

In [ ]:
class AmpDataset(Dataset):
    """把一个 DataFrame(列:aa_seq, AMP)编码成模型输入。"""
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.seqs = list(self.df['aa_seq'])
        self.labels = list(self.df['AMP'].astype(int))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # ProtBERT 要求氨基酸之间用空格分隔
        seq = ' '.join(''.join(self.seqs[idx].split()))
        ids = self.tokenizer(seq, truncation=True, padding='max_length',
                             max_length=self.max_len)
        sample = {k: torch.tensor(v) for k, v in ids.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample

In [ ]:
# 加载分词器(use_fast=False 直接用 vocab.txt 的 WordPiece 分词器,避开新版本的兼容问题)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False, use_fast=False)

In [ ]:
train_dataset = AmpDataset(df, tokenizer)
print('train_dataset 大小:', len(train_dataset))

## 4 · 定义评估指标
准确率、F1、精确率、召回率、MCC、ROC-AUC。

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = logits.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    m = {'accuracy': accuracy_score(labels, preds), 'f1': f1,
         'precision': precision, 'recall': recall,
         'mcc': matthews_corrcoef(labels, preds)}
    if len(np.unique(labels)) == 2:                 # AUC 需要正类概率
        z = logits - logits.max(axis=-1, keepdims=True)
        probs = np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)
        m['roc_auc'] = roc_auc_score(labels, probs[:, 1])
    return m

## 5 · 定义模型
在 ProtBERT-BFD 上接一个二分类头。`model_init` 必须是零参数函数(Trainer 会把超参搜索的 `trial` 传给带参数的版本)。

In [ ]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

## 6 · 配置训练参数
有效批大小 = `per_device_train_batch_size` × `gradient_accumulation_steps` = 1 × 64 = 64,与论文一致。

> 想先快速跑通流程,把 `num_train_epochs` 改成 1。

In [ ]:
OUTPUT_DIR = 'results/amp_bert_train'   # 训练中间产物目录

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,            # 完整复现用 15;冒烟测试用 1
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=64,  # 有效批大小 = 1 × 64
    weight_decay=0.1,
    warmup_steps=0,
    logging_steps=100,
    eval_strategy='no',
    save_strategy='no',
    fp16=True,
    seed=SEED,
    report_to='none',
)

## 7 · 开始训练
完整 15 epoch 在单卡上需要数小时。

In [ ]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

## 8 · 在训练集上自查

In [ ]:
_, _, metrics = trainer.predict(train_dataset)
metrics

## 9 · 保存 AMP-BERT
默认存到 Colab 本地磁盘,**断开连接会丢失**。要长期保留,先挂载 Google Drive,再把 `MODEL_DIR` 改成 Drive 路径(如 `/content/drive/MyDrive/amp_bert`)。

In [ ]:
MODEL_DIR = 'models/amp_bert'
trainer.save_model(MODEL_DIR)
print('已保存到', MODEL_DIR)